In [1]:
import SimpleITK as sitk
from tqdm.notebook import tqdm
from src.data_utils import download_data, build_paths_df

In [2]:
# Путь к датасету
dataset_dir = download_data()

In [3]:
# Формирование датафрейма из путей к файлам
paths_df = build_paths_df(dataset_dir)

In [4]:
def register_flair_to_dwi(dwi_path, flair_path, output_path):
    """Функция, регистрирующая FLAIR изображения к DWI"""

    # Открытие изображений
    fixed = sitk.ReadImage(str(dwi_path), sitk.sitkFloat32)
    moving = sitk.ReadImage(str(flair_path), sitk.sitkFloat32)

    # Первичные совмещение изображений с защитой от изменения размеров и формы областей мозга
    initial_transform = sitk.CenteredTransformInitializer(
        fixed, moving,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.MOMENTS
    )

    # Инициализация регистратора
    registration = sitk.ImageRegistrationMethod()

    # Совмещение на основе статистической взаимосвязи интенсивности изображений
    registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)

    # Настройка способа выбора вокселей для оценки сходства
    registration.SetMetricSamplingStrategy(registration.RANDOM)
    registration.SetMetricSamplingPercentage(1.0)

    # Интерполяция интенсивности вокселей при смещении
    registration.SetInterpolator(sitk.sitkLinear)

    # Оптимизатор смещения изображения
    registration.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0,
        minStep=0.001,
        numberOfIterations=300,
        gradientMagnitudeTolerance=1e-6
    )

    # Масштабирование коэффициентов трансформации для сдвигов и поворотов
    registration.SetOptimizerScalesFromPhysicalShift()

    # Сжатие и размытие изображений для более быстрой начальной подгонки
    registration.SetShrinkFactorsPerLevel([4, 2, 1])
    registration.SetSmoothingSigmasPerLevel([2, 1, 0])
    registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    # Передача оптимизатору первичного совмещения изображений
    registration.SetInitialTransform(initial_transform, inPlace=False)

    # Запуск регистрации и получении матрицы трансформаций
    final_transform = registration.Execute(fixed, moving)

    # Преобразование FLAIR по полученной матрице трансформаций
    registered = sitk.Resample(
        moving, fixed, final_transform,
        sitk.sitkLinear, 0.0, moving.GetPixelID()
    )

    # Сохранение преобразованного изображения
    sitk.WriteImage(registered, str(output_path))

In [5]:
# Список для сохранения ошибок
errors = []

# Проход по каждому пациенту
for patient in tqdm(paths_df['patient_id'].unique()):
    try:

        # Получение путей к DWI и FLAIR снимкам
        dwi_path = paths_df[(paths_df['patient_id'] == patient) & (paths_df['label'] == 'dwi')]['file_path'].item()
        flair_path = paths_df[(paths_df['patient_id'] == patient) & (paths_df['label'] == 'flair')]['file_path'].item()

        # Задаем сохранение регистрированного изображения в папке с исходным
        output_dir = flair_path.parent
        output_path = output_dir / f'{patient}_register_flair.nii'

        # Регистрируем FLAIR к DWI
        register_flair_to_dwi(dwi_path, flair_path, output_path)

    # Если возникла ошибка, сохраняем ее текст
    except Exception as e:
        errors.append((patient, str(e)))

print(f'Ошибок: {len(errors)}')

  0%|          | 0/250 [00:00<?, ?it/s]

Ошибок: 0
